# Ruta óptima para MegaMarket

Jacobo Argote, Andrés Cardona, Miguel Valencia y Juan José Gil

##**CODIGO GENERADO CON CLAUDE**

In [ ]:
"""MegaMarket - Optimización de rutas con A* (versión simplificada) - JJGG.ipynb

Automatically generated by Colab.

Original file is located at
    (pega aquí el link de tu notebook una vez lo subas a Colab)

## Contexto del problema

**MegaMarket** despacha su producto Uva Pasa (marca propia Riovalle) desde su
**Bodega Olaya Herrera** (Medellín) hacia sus clientes, que están agrupados en
dos rutas de reparto:

- **Ruta Poblado:** 8 tiendas dentro de la comuna El Poblado.
- **Ruta Valle de Aburrá (Sur):** 4 CEDIs en Itagüí, La Estrella y alrededores.

Cada ruta es un solo viaje: sale de la bodega, visita a todos sus clientes y
termina en la **Plataforma Logística** (un parqueadero en Guayabal que usan
las dos rutas como punto final, en vez de volver a la bodega).

## Notación (la misma vista en clase)

- `g(n)`: costo real acumulado desde el origen hasta el nodo `n`.
- `h(n)`: heurística, una estimación de lo que falta hasta el destino.
- `f(n) = g(n) + h(n)`: costo total estimado; ordena la cola de prioridad.

## ¿Cómo calculamos la heurística h(n)?

En el ejemplo de clase la heurística era un número fijo por nodo, porque solo
había un destino ("Cliente"). Acá cada ruta tiene **varios clientes distintos**
y ahora también un destino final compartido, así que en vez de escribir un
número a mano por cada combinación posible, calculamos h(n) como la
**distancia en línea recta** entre las coordenadas del nodo actual y las del
destino (fórmula de *haversine*). Es exactamente la misma idea de "distancia
en línea recta hasta el objetivo" que vimos en clase, solo que calculada a
partir de coordenadas reales en vez de estar escrita a mano.

Como la distancia en línea recta nunca puede ser mayor que la distancia real
por las calles, h(n) es **admisible** y A* garantiza la ruta óptima.
"""

from heapq import heappush, heappop
from math import radians, sin, cos, asin, sqrt

# =============================================================================
# 1. COORDENADAS (lat, lon) de cada punto real de la operación
# =============================================================================
COORDENADAS = {
    "Bodega Olaya Herrera": (6.2197, -75.5905),

    # --- Accesos viales a cada zona ---
    "Acceso Poblado (Aguacatala)": (6.1935, -75.5790),
    "Acceso Sur (Itagüí Centro)": (6.1720, -75.6120),

    # --- Ruta Poblado: 8 clientes ---
    "Fruver del Río": (6.2218, -75.5745),
    "Fruver Visitación": (6.2013, -75.5598),
    "Five Sisters": (6.2077, -75.5658),
    "Supermercado Merka 10": (6.2093, -75.5688),
    "Hotel Four Points": (6.2029, -75.5722),
    "Mercados Verona": (6.2046, -75.5605),
    "EAFIT La Bodeguita": (6.2006, -75.5783),
    "Redetrans": (6.2112, -75.5628),

    # --- Ruta Valle de Aburrá (Sur): 4 clientes ---
    "CEDI Vaquita Itagüí": (6.1735, -75.6152),
    "CEDI Euro La Estrella": (6.1448, -75.6198),
    "CEDI Merkepaisa": (6.1618, -75.6062),
    "CEDI Supermú": (6.1548, -75.6122),

    # --- Destino final compartido por las dos rutas ---
    # Cra. 51C #12B Sur-76, San Fernando, Guayabal, Medellín.
    # OJO: esta coordenada es una ESTIMACIÓN a partir de la nomenclatura vial
    # (mismo método y mismo margen de error, +/- 200-400 m, que se usó para
    # el resto de puntos del proyecto original). Si tienes la ubicación
    # exacta (por ejemplo, del pin en Google Maps), reemplázala aquí.
    "Plataforma Logística": (6.1976, -75.5807),
}

# =============================================================================
# 2. HEURÍSTICA h(n): distancia en línea recta (haversine)
# =============================================================================
def heuristica(nodo, destino):
    """Distancia en línea recta entre dos lugares, en kilómetros.

    Es el mismo concepto de "distancia en línea recta hasta el objetivo"
    que vimos en clase, calculado a partir de las coordenadas reales.
    """
    lat1, lon1 = COORDENADAS[nodo]
    lat2, lon2 = COORDENADAS[destino]
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    aux = sin(d_lat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(d_lon / 2) ** 2
    return 2 * 6371.0 * asin(sqrt(aux))  # 6371 km = radio de la Tierra


# =============================================================================
# 3. GRAFOS DE CADA RUTA
#    Igual que en clase: diccionario {nodo: [(vecino, costo_km), ...]}.
#    El camión primero llega al acceso vial de la zona y desde ahí reparte
#    a cada cliente. Las calles se recorren en los dos sentidos, así que
#    cada conexión se agrega también "de regreso" (por eso usamos la
#    función conectar() en vez de escribir cada arista dos veces a mano).
# =============================================================================
def conectar(red, a, b, costo):
    """Agrega una conexión en AMBOS sentidos (a->b y b->a), porque una calle
    se puede recorrer de ida y de vuelta."""
    red.setdefault(a, []).append((b, costo))
    red.setdefault(b, []).append((a, costo))


red_poblado = {}
conectar(red_poblado, "Bodega Olaya Herrera", "Acceso Poblado (Aguacatala)", 3.2)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Fruver del Río", 3.2)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Fruver Visitación", 2.3)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Five Sisters", 2.2)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Supermercado Merka 10", 2.1)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Hotel Four Points", 1.3)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Mercados Verona", 2.4)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "EAFIT La Bodeguita", 0.8)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Redetrans", 2.7)

red_valle_aburra = {}
conectar(red_valle_aburra, "Bodega Olaya Herrera", "Acceso Sur (Itagüí Centro)", 5.8)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Vaquita Itagüí", 0.4)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Euro La Estrella", 3.2)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Merkepaisa", 1.3)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Supermú", 1.9)

# --- Conectamos la Plataforma Logística a cada ruta ---
# En vez de escribir el km a mano, usamos heuristica() (línea recta entre
# coordenadas) para calcular la distancia al punto de la ruta más cercano.
# Poblado: se conecta por su acceso vial (Aguacatala). Valle de Aburrá: se
# conecta por su acceso vial (Itagüí Centro).
conectar(
    red_poblado, "Acceso Poblado (Aguacatala)", "Plataforma Logística",
    round(heuristica("Acceso Poblado (Aguacatala)", "Plataforma Logística"), 1),
)
conectar(
    red_valle_aburra, "Acceso Sur (Itagüí Centro)", "Plataforma Logística",
    round(heuristica("Acceso Sur (Itagüí Centro)", "Plataforma Logística"), 1),
)

# Clientes de cada ruta, para poder recorrerlos con un for
clientes_poblado = [
    "Fruver del Río", "Fruver Visitación", "Five Sisters",
    "Supermercado Merka 10", "Hotel Four Points", "Mercados Verona",
    "EAFIT La Bodeguita", "Redetrans",
]

clientes_valle_aburra = [
    "CEDI Vaquita Itagüí", "CEDI Euro La Estrella",
    "CEDI Merkepaisa", "CEDI Supermú",
]

# --- Conexiones directas entre tiendas cercanas de la misma ruta ---
# Hasta acá, el único camino entre dos tiendas es pasando por el acceso
# vial de la zona. En la práctica, si dos tiendas quedan muy cerca, el
# camión va de una a la otra directo, sin devolverse a la vía principal.
# Igual que en el proyecto original: conectamos los pares que quedan a
# menos de UMBRAL_CERCANIA_KM en línea recta.
UMBRAL_CERCANIA_KM = 1.5


def conectar_clientes_cercanos(red, clientes, umbral=UMBRAL_CERCANIA_KM):
    for i in range(len(clientes)):
        for j in range(i + 1, len(clientes)):
            a, b = clientes[i], clientes[j]
            distancia = heuristica(a, b)
            if distancia < umbral:
                conectar(red, a, b, round(distancia, 2))


conectar_clientes_cercanos(red_poblado, clientes_poblado)
conectar_clientes_cercanos(red_valle_aburra, clientes_valle_aburra)

# =============================================================================
# 4. A* (A-Star) — misma estructura vista en clase
# =============================================================================
def a_estrella(red, origen, destino):
    # Inicializamos la cola de prioridad con: (f(n), g(n), camino)
    # f(n) = g(n) + h(n) -> Costo Total Proyectado
    cola = [(heuristica(origen, destino), 0, [origen])]
    visitados = set()

    while cola:
        # Extraemos el nodo con el menor f(n) (el más prometedor)
        f, g, camino = heappop(cola)
        nodo_actual = camino[-1]

        # Si el nodo actual es el destino, hemos terminado con éxito
        if nodo_actual == destino:
            return camino, g

        if nodo_actual not in visitados:
            visitados.add(nodo_actual)

            # Exploramos las conexiones del nodo actual
            for vecino, costo in red[nodo_actual]:
                # g(n) es el costo acumulado real desde el origen
                nuevo_g = g + costo
                # f(n) = g acumulado + estimación h del vecino
                nuevo_f = nuevo_g + heuristica(vecino, destino)

                heappush(cola, (nuevo_f, nuevo_g, camino + [vecino]))

    return None, None


# =============================================================================
# 5. RECORRIDO COMPLETO DE REPARTO
#    El camión sale UNA sola vez de la bodega, visita a todos los clientes
#    de la ruta y termina en destino_final. En cada paso usamos A* para ir
#    del punto donde estamos hasta el cliente sin visitar más cercano
#    ("vecino más cercano"): es una estrategia sencilla, no prueba todas
#    las combinaciones posibles, así que no garantiza el orden más corto de
#    TODOS los órdenes posibles (para eso habría que probar combinaciones,
#    que es justo la parte que dejamos por fuera para no complicar el
#    ejercicio).
# =============================================================================
def recorrido_completo(red, origen, clientes, destino_final=None):
    """Recorrido de reparto: sale de 'origen', visita cada cliente una sola
    vez (siempre yendo al más cercano desde donde está parado) y al final
    llega a 'destino_final'.

    Si no se indica destino_final, el camión regresa al punto de origen
    (comportamiento por defecto). Si se indica un punto distinto (por
    ejemplo la Plataforma Logística), el recorrido termina ahí en vez de
    volver a la bodega.
    """
    if destino_final is None:
        destino_final = origen

    pendientes = clientes.copy()
    paradas = [origen]
    orden_clientes = []  # en qué orden se visitó cada cliente (para numerarlos)
    posicion_actual = origen
    distancia_total = 0

    while pendientes:
        # Entre los clientes que faltan, buscamos el más cercano desde
        # donde estamos parados ahora mismo (no siempre desde la bodega)
        mejor_cliente, mejor_camino, mejor_costo = None, None, float("inf")
        for cliente in pendientes:
            camino, costo = a_estrella(red, posicion_actual, cliente)
            if costo < mejor_costo:
                mejor_cliente, mejor_camino, mejor_costo = cliente, camino, costo

        paradas.extend(mejor_camino[1:])  # evitamos repetir el nodo de partida
        orden_clientes.append(mejor_cliente)
        distancia_total += mejor_costo
        posicion_actual = mejor_cliente
        pendientes.remove(mejor_cliente)

    # Último tramo: hacia destino_final (la bodega, por defecto, o el
    # punto final que se haya indicado)
    camino_final, costo_final = a_estrella(red, posicion_actual, destino_final)
    paradas.extend(camino_final[1:])
    distancia_total += costo_final

    return paradas, distancia_total, orden_clientes


# =============================================================================
# 6. EJECUCIÓN
# =============================================================================
if __name__ == "__main__":
    for nombre_ruta, red, clientes in [
        ("POBLADO", red_poblado, clientes_poblado),
        ("VALLE DE ABURRÁ (SUR)", red_valle_aburra, clientes_valle_aburra),
    ]:
        print("#" * 60)
        print(f"RUTA {nombre_ruta}")
        print("#" * 60)

        print("\n-- Recorrido completo de reparto (Bodega -> clientes -> Plataforma Logística) --")
        paradas, distancia_total, orden_clientes = recorrido_completo(
            red, "Bodega Olaya Herrera", clientes, destino_final="Plataforma Logística"
        )
        print(" → ".join(paradas))
        print(f"Distancia total del recorrido: {distancia_total:.1f} km")
        print()

############################################################
RUTA POBLADO
############################################################

-- Recorrido completo de reparto (Bodega -> clientes -> Plataforma Logística) --
Bodega Olaya Herrera → Acceso Poblado (Aguacatala) → EAFIT La Bodeguita → Hotel Four Points → Supermercado Merka 10 → Five Sisters → Redetrans → Mercados Verona → Fruver Visitación → Acceso Poblado (Aguacatala) → Fruver del Río → Acceso Poblado (Aguacatala) → Plataforma Logística
Distancia total del recorrido: 16.8 km

############################################################
RUTA VALLE DE ABURRÁ (SUR)
############################################################

-- Recorrido completo de reparto (Bodega -> clientes -> Plataforma Logística) --
Bodega Olaya Herrera → Acceso Sur (Itagüí Centro) → CEDI Vaquita Itagüí → Acceso Sur (Itagüí Centro) → CEDI Merkepaisa → CEDI Supermú → CEDI Euro La Estrella → Acceso Sur (Itagüí Centro) → Plataforma Logística
Distancia total del r